# Debugging {#sec-debugging}

*The best debugging is to avoid introducing in the first place, but no one is perfect...*

When it comes to finding bugs, what separates a bad afternoon from a five-minute fix is not luck, it a is method that can be learned.

1. **Read the traceback from the bottom up.** The last line names the error and usually says what value was involved. Start there, not at the top.
2. **Locate the failing line.** Python tells you the exact line, and the file and function it was in. Find it before you theorize about anything else.
3. **Form a hypothesis.** Given the error and the line, what specific thing must be true about the values at that moment for this error to happen? State it as a sentence you could be wrong about, not a vague feeling.
4. **Confirm it before you fix it.** Print the value you suspect, or step through the line with the codelens widget, or write a small test that isolates just this case. Only once you have seen the actual value that caused the failure do you know your hypothesis was right, and only then do you change the code.

Skipping straight to step four, changing something and hoping, is how a five-minute bug turns into an hour. The method is slower on the first bug and much faster on every one after it, because it is the same four steps every time.

In [ ]:
import sandbox_widget, codelens_widget

## The traceback

When a function is called, Python keeps a record of where it will need to return to when the function as run. Each such record is called a "frame" and as functions call other functions, these records stack up into something called, well, a stack. The stack grows every time a function is called shrinks when a function returns.

When your program fails (an exception is raised), Python prints the state of the stack at that moment. That is the "traceback". It traces the path of calls from the top frame of your program down to the line that failed. The header "Traceback (most recent call last)" means that outermost frame or call is printed first and the innermost frame, where the your program failed, is printed last. So the traceback not only shows you at which the program failed, it also shows you how it got there. Each frame is two lines: the file, line number, and function name, followed by the source code that was executing on that line. At the bottom is you can see which type of exception (error) that occurred and its message. In the example below it says "ZeroDivisionError: division by zero". The type tells you the category of problem, and the message usually names the offending value.

In [ ]:
%%sandbox
def gc_content(dna):
    "Fraction of bases that are G or C."
    gc = dna.count('G') + dna.count('C')
    return gc / len(dna)

sequences = ['ATGC', 'GGCC', '']
results = []
for seq in sequences:
    gc = gc_content(seq)
    results.append(gc)

print(results)

To identify the problem, start at the bottom and read upwards. The bottom frame is where the exception was raised, but it’s not necessarily where the bug is. The error may have been caused by bad data passed in from further up. If the bottom frames are inside the standard library or an installed package, the fault is almost always in the first frame above them that belongs to your own code, since that is where you handed the library something it could not handle. The traceback shows code, not values, so the final step is inference: combine the exception message with the chain of calls to work out what values must have reached the failing line. If that isn’t obvious, a you can always add a `print` statement to print the relevant variables just before the offending line.

In our example, it is line 4 in the bottom frame that failed: `return gc / len(dna)`. "ZeroDivisionError: division by zero" tells us the program failed because we tried to divide a number by zero, which is not allowed. So, for some reason, `len(dna)` evaluates to zero. For that to happen, the length of `dna` must be zero, which again means that `dna` must be an empty string. That was was clearly not our intention, so not the quest is to trace our way upwards through the traceback to figure out how it ended up being an empty string. Form a hypothesis and test it. An obvious one is that one of the strings in `sequences` is empty. Confirm it by looking at the list. That was indeed the case. Now decide how to fix it: If you accidentally added an empty string to your test sequences, you can just remove it. But maybe the bug exposes that your code is not robust enough to handle empty strings when it should be. In that case the fix is to decide what the GC content of an empty sequence even means and what your code should produce in that case. Maybe you want to change your function to something like:

In [ ]:
def gc_content(dna):
    "Fraction of bases that are G or C. Undefined for an empty sequence."
    if len(dna) == 0:
        return None
    gc = dna.count('G') + dna.count('C')
    return gc / len(dna)

## Silent bugs

Run the code below. It does not crash, and `count_start_codons` returns `0`. Decide for yourself, by counting on paper, whether that is actually correct before you read on.

In [ ]:
%%sandbox
def count_start_codons(dna):
    "Count how many times the start codon ATG appears, including overlaps."
    count = 0
    for i in range(0, len(dna), 3):
        codon = dna[i:i+3]
        if codon == 'ATG':
            count += 1
    return count

print(count_start_codons('AATGATGC'))

The bugs that crash are, in one sense, the friendly ones, because Python is shouting at you about exactly where to look. The hard ones are where your code silently something other than what you intended. When there is no error, the "failing line" is not a crash site, it is the line whose *assumption* is wrong. The best you can do is to add print statements to see if the code runs as you assume.

In [ ]:
%%sandbox
def count_start_codons(dna):
    "Count how many times the start codon ATG appears, including overlaps."
    count = 0
    for i in range(0, len(dna), 3):
        print('i:', i)
        codon = dna[i:i+3]
        print('codon:', codon)
        if codon == 'ATG':
            count += 1
            print('found one!')
    return count

print(count_start_codons('AATGATGC'))

I looks like the values of `i` in the for loop takes values 0, 2, 6, and that slicing out codons starting at those indices does not find the ATG codons we know are there. So the arguments passed to the range function is a sensible place to look for a mistake. In this case you probably intended the `i` values to be 0, 1, 2, 3, 4, 5. To achieve that, it should have been `range(len(dna)-1)`. See if that fixes it.

An alternative way to trace the values as the code runs is to use the codelens widget. That would also have revealed that the `i` value jumps by 3 instead of 1.

In [ ]:
%%codelens
def count_start_codons(dna):
    "Count how many times the start codon ATG appears, including overlaps."
    count = 0
    for i in range(0, len(dna), 3):
        codon = dna[i:i+3]
        if codon == 'ATG':
            count += 1
    return count

print(count_start_codons('AATGATGC'))


### Confirming with codelens instead of print

Printing a suspect value works, but sometimes you want to watch a loop's variables change on every single step rather than guess where to insert a print. That is exactly what the codelens widget is for: point it at `count_start_codons('AATGATGC')` with the fix above and step through it one line at a time, watching `i` and `count` in the panel beside the code. Use it whenever your hypothesis is about *how a value changes over several steps* rather than about *what a value is at one point*; a single print statement cannot show you a sequence of changes the way stepping through the loop can.



#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Here is a third planted bug. Use the four-step method, in order, and write down your hypothesis in a sentence before you attempt any fix.


In [ ]:
import codelens_widget

In [ ]:
%%codelens
def reverse_complement(dna):
    "Return the reverse complement of a DNA strand."
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    result = ''
    for base in dna:
        result = complement[base] + result
    return result

print(reverse_complement('ATGC'))
print(reverse_complement('atgc'))


The first call is correct. The second is not, and it fails the same quiet way bug two did: no traceback at all. Find the failing line, form the hypothesis, confirm it by printing the one value that matters, and only then fix it.

For your logbook this week, describe a bug from your own project code this week that you found using this four-step method rather than by guessing, and say which step actually found it.